# 05 · Retraining on real data

**Purpose.** The recipe for moving from the twin to the prototype: same code, real data.

```
 MQTT vawt/sensors ─► store ─► export month folders in the dataset layout ─► python -m aerosolar.ml.train ─► evaluate
 Open-Meteo every hour ─► archive ─► forecast_issued.parquet
```
**Steps**
1. **From day one, archive the Open-Meteo forecast every hour** (`aerosolar.ml.forecast_day.fetch_open_meteo`). Without the
   forecasts that were *issued*, no fair day-ahead model can be trained.
2. Calibrate the configs (CALIBRATION.md), so the twin matches the hardware.
3. Export each month to `data/<name>/` with `telemetry_1hz.parquet`, `status_1hz.parquet`, `forecast_issued.parquet`, `meta.json`.
4. Point `aerosolar/ml/campaign.py` at the real folders: earliest months = train (keep the last 5 days of each month for
   calibration), the latest month(s) = test. Never evaluate on months used for fitting.
5. Until a full year exists, train on *simulated year + real months* and weight the real months up; the twin then fills the
   seasons not yet seen.
6. Accept the model when the real test month meets the same statistics as in `reports/forecast_evaluation.html`.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerosolar.config import Config, REPO_ROOT
from aerosolar.sim.realism import _style, S1, S2, S3, INK, INK2
_style(); plt.rcParams['figure.dpi'] = 110
cfg = Config.load()
DATA = REPO_ROOT / 'data'
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 30)

In [2]:
from aerosolar.ml.forecast_day import fetch_open_meteo
try:
    fc = fetch_open_meteo(cfg); print(len(fc), 'hours fetched'); display(fc.head())
except Exception as e:
    print('Open-Meteo not reachable from here:', type(e).__name__)

48 hours fetched


,issue_time,valid_time,wind_speed_10m,shortwave_radiation,temperature_2m,lead_h
0,2026-10-01 13:00:00+00:00,2026-10-01 14:00:00+00:00,5.31,67.0,35.2,1
1,2026-10-01 13:00:00+00:00,2026-10-01 15:00:00+00:00,2.68,0.0,35.0,2
2,2026-10-01 13:00:00+00:00,2026-10-01 16:00:00+00:00,4.90,0.0,33.9,3
3,2026-10-01 13:00:00+00:00,2026-10-01 17:00:00+00:00,5.86,0.0,32.5,4
4,2026-10-01 13:00:00+00:00,2026-10-01 18:00:00+00:00,5.64,0.0,32.0,5


**What to notice.** The live forecast arrives in exactly the schema the model was trained on, so the switch is a data
source change, not a code change.

> **Caution.** Real forecast errors will differ from the simulated ones; re-run the evaluation on real months before trusting
> the stated accuracy.

## References
* Romano, Patterson, Candès (2019). *Conformalized quantile regression.* NeurIPS.
* Ke et al. (2017). *LightGBM: a highly efficient gradient boosting decision tree.* NeurIPS.
* Yang et al. (2020). *Verification of deterministic solar forecasts.* Solar Energy 210 (clear-sky index, smart persistence, skill scores).
* Hong et al. (2016). *Probabilistic energy forecasting: GEFCom2014 and beyond.* IJF 32.
* Haurwitz (1945). *Insolation in relation to cloudiness and cloud density.* J. Meteorology.
* Kamoji, Kedare, Prabhu (2009). *Performance tests on helical Savonius rotors.* Renewable Energy 34.